In [10]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
import joblib

# nltk.download('stopwords')
# nltk.download('wordnet')
# nltk.download('omw-1.4')

# load dataset
df = pd.read_csv("test.csv", encoding='latin-1')

df = df[['text', 'sentiment']]
df['label'] = df['sentiment'].map({'negative': 0, 'neutral': 1, 'positive': 2})

# set stopwords and lemmatizer
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    words = text.split()
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    return ' '.join(words)

# clean text
df['clean_text'] = df['text'].apply(clean_text)

# split into cols
X = df['clean_text']
y = df['label']

# vectorizer
vectorizer = TfidfVectorizer(max_features=5000)
X_tfidf = vectorizer.fit_transform(X)

# apply balancing SMOTE
smote = SMOTE(sampling_strategy='auto', random_state=42)
X_res, y_res = smote.fit_resample(X_tfidf, y)

# model set krle bhai
model = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

# cross validation score for 10 folds
cross_val_scores = cross_val_score(model, X_res, y_res, cv=10, scoring='accuracy')

print("Cross-Validation Accuracy Scores (10-fold):", cross_val_scores)
print("Mean Accuracy: ", cross_val_scores.mean())

model.fit(X_res, y_res)

y_pred = model.predict(X_tfidf)

# save
joblib.dump(model, 'sentiment_model.pkl')
joblib.dump(vectorizer, 'vectorizer.pkl')

print("\nModel and Vectorizer Saved Successfully!")


Cross-Validation Accuracy Scores (10-fold): [0.68591224 0.69053118 0.67667436 0.66974596 0.69053118 0.68360277
 0.66512702 0.80138568 0.87297921 0.85185185]
Mean Accuracy:  0.7288341459242152

Model and Vectorizer Saved Successfully!


In [12]:
import joblib
from sklearn.metrics import accuracy_score
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re
import nltk

# load model and vectorizer
model = joblib.load('sentiment_model.pkl')
vectorizer = joblib.load('vectorizer.pkl')

# sample texts and labels
test_texts = [
    "I love this product!",                 # Positive
    "It’s okay, nothing special.",          # Neutral
    "Worst service ever.",                  # Negative
    "Absolutely fantastic experience.",     # Positive
    "Not bad, could be better.",            # Neutral
    "I’m very disappointed in the quality.",# Negative
    "This is the best I’ve ever used!",     # Positive
    "Mediocre and uninspiring.",            # Neutral
    "Terrible. Will never buy again.",      # Negative
    "Great customer support and fast delivery.",  # Positive
    "It was alright, nothing great.",       # Neutral
    "The product broke after one use.",     # Negative
    "Amazing! Five stars!",                 # Positive
    "Fine. Just fine.",                     # Neutral
    "Horrible packaging and rude staff."    # Negative
]

# Labels: 2 = Positive, 1 = Neutral, 0 = Negative
true_labels = [
    2, 1, 0, 2, 1, 0, 2, 1, 0, 2,
    1, 0, 2, 1, 0
]

def clean_text(text):
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    words = text.split()
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    return ' '.join(words)

# clean input and predict
cleaned_texts = [clean_text(text) for text in test_texts]
X_test = vectorizer.transform(cleaned_texts)
predictions = model.predict(X_test)

# Output
print("Predictions:", predictions)
print("Accuracy on test samples:", accuracy_score(true_labels, predictions))


Predictions: [2 1 0 2 0 0 2 1 1 2 2 0 2 1 0]
Accuracy on test samples: 0.8


In [14]:
import joblib
from sklearn.metrics import accuracy_score
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re
import nltk

# load model and vectorizer
model = joblib.load('sentiment_model.pkl')
vectorizer = joblib.load('vectorizer.pkl')

# sample texts and labels
test_texts = [
    "I can't believe I missed the deadline! Ugh, so stressed out. 😤",
    "This game keeps freezing, really annoying. 😡",
    "Had an awful experience with customer service today. Will never return.",
    "Feeling drained. Can’t seem to catch a break this week. 😞",
    "Horrible weather today. It's cold and rainy all day. 🌧️",
    "Just had lunch, now back to work.",
    "Need to schedule a dentist appointment soon. Nothing exciting.",
    "I saw a good movie last night, but it was nothing special.",
    "Started reading a new book today, still early to tell how it is.",
    "The train was late, but that's not unusual for Monday mornings.",
     "Had the best day ever at the beach! 🌊🌞 #blessed",
    "Just finished reading a book that changed my life. Highly recommend it! 📚",
    "I’m feeling fantastic today! Everything’s going right. 😄",
    "Got an amazing deal on a new laptop! Can't wait to start using it. 💻💯",
    "The weather is perfect for a walk in the park. So relaxing! 🌳",
    "Hi, Im Mohsin",
]

# Labels: 2 = Positive, 1 = Neutral, 0 = Negative
true_labels = [
    0, 0, 0, 0, 0, 1, 1, 1, 1, 1,
    2, 2, 2, 2, 2, 2
]

def clean_text(text):
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    words = text.split()
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    return ' '.join(words)

# clean input and predict
cleaned_texts = [clean_text(text) for text in test_texts]
X_test = vectorizer.transform(cleaned_texts)
predictions = model.predict(X_test)

# Output
print("Predictions:", predictions)
print("Accuracy on test samples:", accuracy_score(true_labels, predictions))


Predictions: [0 0 0 0 0 1 1 2 1 1 2 2 2 2 2 2]
Accuracy on test samples: 0.9375


In [16]:
import re
import nltk
import joblib
import gradio as gr
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    words = text.split()
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    return ' '.join(words)

# Load model and vectorizer
model = joblib.load('sentiment_model.pkl')
vectorizer = joblib.load('vectorizer.pkl')

# Prediction function
def predict_sentiment(tweet):
    cleaned = clean_text(tweet)
    vectorized = vectorizer.transform([cleaned])
    prediction = model.predict(vectorized)[0]
    sentiment_map = {0: 'Negative 😔', 1: 'Neutral 😐', 2: 'Positive 😊'}
    return f"Predicted Sentiment:   {sentiment_map[prediction]}"

# Gradio UI
interface = gr.Interface(
    fn=predict_sentiment,
    inputs=gr.Textbox(lines=4, placeholder="Enter a tweet here..."),
    outputs="text",
    title="Tweet Sentiment Analyzer"
)

interface.launch(share=True)  # `share=True` gives you a public link

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://517a28b6c9bdb52ea2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
